# Loss Functions

In this exercise, you will compare the effects of Loss functions on a `LinearRegression` model.

👇 Let's download a CSV file to use for this challenge and convert it into a DataFrame

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
741,0.76,661.5,416.5,122.5,7.0,0.40,39.760
732,0.82,612.5,318.5,147.0,7.0,0.40,29.965
507,0.74,686.0,245.0,220.5,3.5,0.25,13.025
445,0.82,612.5,318.5,147.0,7.0,0.25,27.195
426,0.64,784.0,343.0,220.5,3.5,0.25,18.560


🎯 Your task is to predict the average temperature inside a greenhouse based on its design. Your temperature predictions will help you choose the appropriate greenhouse design based on the climate needs of each plant.

🌿 You know that plants can tolerate small temperature variations, but they become exponentially more sensitive as temperature variations increase.

## 1. Theory

❓ Theoretically, which Loss function would you train your model on to limit the risk of killing the plants?

<details>
<summary> 🆘 Answer </summary>
    
Theoretically, you would use the Mean Squared Error (MSE) Loss function. This penalizes outlier predictions and prevents your model from making large errors. This will result in smaller temperature variations and lower risk for the plants.

</details>

MSE LF

## 2. Practice

### 2.1 Preprocessing

❓ Standardize the features

In [6]:
from sklearn.preprocessing import StandardScaler

X = data.loc[:,'Relative Compactness':'Glazing Area']
y = data["Average Temperature"]

scaler = StandardScaler().fit(X)

X_scaled = scaler.transform(X)

### 2.2 Modelling

In this section, you will verify the theory by evaluating models optimized on different Loss functions.

### Ordinary Least Squares (MSE) Loss

❓ **10-Fold Cross-validate** a Linear Regression model optimized with **Stochastic Gradient Descent** (SGD) on the **Ordinary Least Squares Loss** (MSE)

In [15]:
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import make_scorer, max_error

scoring = {
    "r2": "r2",
    "max_error": make_scorer(max_error, greater_is_better=False)
}

sgd_mse = SGDRegressor(loss="squared_error")

sgd_mse_cv_results = cross_validate(sgd_mse, X_scaled, y, cv=10, scoring=scoring)

sgd_mse_cv_results

{'fit_time': array([0.00612926, 0.00547004, 0.00485682, 0.00480008, 0.00472713,
        0.00386715, 0.00486898, 0.00412488, 0.00479507, 0.00422812]),
 'score_time': array([0.0007689 , 0.00141311, 0.00062108, 0.00073099, 0.00061798,
        0.00054789, 0.00050116, 0.00047708, 0.00061488, 0.00050998]),
 'test_r2': array([0.78762059, 0.90883354, 0.89543348, 0.88394014, 0.93136558,
        0.89664921, 0.92718569, 0.91590225, 0.89462628, 0.94067608]),
 'test_max_error': array([-9.81839704, -8.60291902, -8.77821828, -9.22457737, -8.8525566 ,
        -8.64584259, -8.50400348, -8.84787211, -8.3565554 , -7.6808095 ])}

❓ Compute:
- The mean cross-validated R2 score and store it in a variable `r2`
- The largest single prediction error in °C across all your folds and store it in `max_error_celsius`

(Hint: `max_error` is an accepted scoring metric in sklearn)

In [16]:
r2 = sgd_mse_cv_results["test_r2"].mean()
r2

0.8982232827986311

In [18]:
max_error_celsius = abs(sgd_mse_cv_results["test_max_error"].min())
max_error_celsius

9.81839703818601

### Mean Absolute Error (MAE) Loss

What happens if we optimize our model on MAE instead?

❓ **10-Fold Cross-validate** a Linear Regression model optimized with **Stochastic Gradient Descent** (SGD) on the **MAE** Loss

<details>
<summary>💡 Hints</summary>

- MAE loss cannot be directly specified in `SGDRegressor`. It needs to be designed by setting the right parameters

</details>

In [21]:
mae_sgd = SGDRegressor(loss="epsilon_insensitive", epsilon = 0)

scoring = {
    "r2": "r2",
    "max_error": make_scorer(max_error, greater_is_better=False)
}

mae_sgd_cv_results = cross_validate(
    mae_sgd, 
    X_scaled, 
    y, 
    cv = 10,  
    scoring = scoring
)

❓ Compute:
- The mean cross-validated R2 score, store it in `r2_mae`
- The largest single prediction error across all your folds, store it in `max_error_mae`

In [22]:
r2_mae = mae_sgd_cv_results["test_r2"].mean()
r2_mae

0.8760689358589536

In [23]:
max_error_mae = abs(mae_sgd_cv_results["test_max_error"].min())
max_error_mae

11.198481963397018

## 3. Conclusion

❓ Which of the models you evaluated seems most suitable for your task?

<details>
<summary> 🆘 Answer </summary>
    
Although the mean cross-validated r2 scores between the two models are approximately similar, the model optimized on MAE has a higher chance of occasionally making larger errors, which increases the risk of killing the plants!
    
</details>

MAE

# 🏁 Check your code and submit your notebook

In [24]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/loss-functions/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.11s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

